# Exp6 Prob-MOEA/D (2022)

Runs the uploaded Prob-MOEA/D baseline method: SurrogateKriging + ProbMOEAD_v3 with population size 50 and 10000 total evaluations.

Offline dataset settings are loaded from `experiments/config.yaml`, using `max(11 * n_var - 1, 100)` points and the Exp1 training seed. Each method initializes its population with the same number of leading offline points as its configured population size.


### Package


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
import types
from pathlib import Path

sys.dont_write_bytecode = True
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"paramz(\..*)?")
warnings.filterwarnings("ignore", message=r"Failed to config .* module.*", category=UserWarning)
warnings.filterwarnings("ignore", message=r"Gym has been unmaintained.*", category=UserWarning)

if "imp" not in sys.modules and importlib.util.find_spec("imp") is None:
    sys.modules["imp"] = types.ModuleType("imp")

DEPENDENCIES = {
    "pymoo": "pymoo==0.6.1.6",
    "pyDOE2": "pyDOE2",
    "GPy": "GPy",
    "yaml": "pyyaml",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "diversipy": "diversipy",
    "optproblems": "optproblems",
    "graphviz": "graphviz",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
}

if "Prob_MOEAD" == "DDMOEA_GAN":
    DEPENDENCIES.update({"torch": "torch"})

for import_name, pip_name in DEPENDENCIES.items():
    try:
        importlib.import_module(import_name)
        print(f"{import_name} is available.")
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pip_name])

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')

repo_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/content/drive/MyDrive/2026 Real-wrold problem'),
    Path('/rds/projects/w/wangsu-building-automation/Huanbo/2026_real_world_problem'),
]
for repo_root in repo_candidates:
    if (repo_root / 'src').exists() and (repo_root / 'experiments' / 'baseline' / 'batch_experiments.py').exists():
        repo_root_string = str(repo_root)
        while repo_root_string in sys.path:
            sys.path.remove(repo_root_string)
        sys.path.insert(0, repo_root_string)
        baseline_root = repo_root / 'experiments'
        while str(baseline_root) in sys.path:
            sys.path.remove(str(baseline_root))
        sys.path.insert(0, str(baseline_root))
        print(f"Using repository root: {repo_root}")
        break
else:
    raise FileNotFoundError('Could not locate current repository root with src and experiments/baseline.')

importlib.invalidate_caches()
sys.modules.pop('baseline.batch_experiments', None)
sys.modules.pop('baseline', None)
import baseline.batch_experiments as batch_experiments

print(f"Loaded baseline runner: {batch_experiments.__file__}")
run_suite = batch_experiments.run_prob_moead_suite


### Run


In [ ]:
all_results = run_suite()


### Gap improvement table


In [ ]:
# Per-seed results and aggregate summaries are written by the suite.
